# Ensemble Methods: Bagging, Boosting, and Stacking
## Learning Objectives
- Implement bagging from scratch (Random Forest)
- Understand AdaBoost and Gradient Boosting mechanics
- Build a stacking ensemble with a meta-learner


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Bagging from scratch
class BaggingClassifier:
    def __init__(self, n_estimators=20, max_samples=0.8, random_state=42):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.random_state = random_state
        self.estimators = []

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n = int(len(X) * self.max_samples)
        for _ in range(self.n_estimators):
            idx = rng.choice(len(X), size=n, replace=True)
            tree = DecisionTreeClassifier(max_depth=5, random_state=rng.randint(1000))
            tree.fit(X[idx], y[idx])
            self.estimators.append(tree)

    def predict(self, X):
        preds = np.array([e.predict(X) for e in self.estimators])
        return np.apply_along_axis(lambda x: np.bincount(x.astype(int)).argmax(), 0, preds)

bag = BaggingClassifier(n_estimators=50)
bag.fit(X_train, y_train)
print(f'Bagging accuracy: {accuracy_score(y_test, bag.predict(X_test)):.4f}')

single_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
single_tree.fit(X_train, y_train)
print(f'Single tree:      {accuracy_score(y_test, single_tree.predict(X_test)):.4f}')

In [ ]:
# Gradient Boosting intuition
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

models = {
    'Random Forest (Bagging)': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost':                AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':       GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Logistic Regression':     LogisticRegression(max_iter=1000)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = acc
    print(f'  {name:30s}: {acc:.4f}')

In [ ]:
# Stacking ensemble from scratch
from sklearn.model_selection import cross_val_predict

base_models = [
    ('rf',  RandomForestClassifier(n_estimators=50, random_state=42)),
    ('gb',  GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('lr',  LogisticRegression(max_iter=1000))
]

# Generate out-of-fold predictions (meta-features)
meta_train = np.column_stack([
    cross_val_predict(clf, X_train, y_train, cv=5, method='predict_proba')[:, 1]
    for _, clf in base_models
])

# Train meta-learner
for _, clf in base_models:
    clf.fit(X_train, y_train)

meta_test = np.column_stack([
    clf.predict_proba(X_test)[:, 1]
    for _, clf in base_models
])

meta_learner = LogisticRegression()
meta_learner.fit(meta_train, y_train)
stack_acc = accuracy_score(y_test, meta_learner.predict(meta_test))
print(f'\nStacking ensemble accuracy: {stack_acc:.4f}')

## Exercises
1. Implement AdaBoost from scratch using weighted resampling.
2. Compare XGBoost, LightGBM, and CatBoost on the same dataset.
3. Visualize the learning curves for different numbers of estimators.


## Summary
- Bagging reduces variance by averaging many high-variance models.
- Boosting reduces bias by sequentially correcting errors.
- Stacking learns which base models to trust with a meta-learner.
